In [1]:
# %% [markdown]
# # Ray Tune 核心参数影响分析脚本
# 重点关注参数：head_hidden_dims, conflict_alpha, weight_decay 对最佳结果的影响。

# %%
import os
import json
import pandas as pd

# %%
# 定义基础搜索路径
base_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha_head/exp_v10_mix_both_head_mean"

print(f"开始解析路径: {base_path}")

# %%
all_trials_data = []

# 遍历目录
for root, dirs, files in os.walk(base_path):
    if "params.json" in files and "result.json" in files:
        params_path = os.path.join(root, "params.json")
        result_path = os.path.join(root, "result.json")
        
        # 1. 解析参数
        try:
            with open(params_path, 'r', encoding='utf-8') as f:
                params = json.load(f)
        except Exception as e:
            continue
            
        # 2. 解析实验日志结果，寻找所有 epoch 中最好的 local_best_test_Target_Y_AUUC
        best_auuc = -float('inf')
        best_epoch_metrics = None
        
        try:
            with open(result_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip():
                        continue
                    epoch_data = json.loads(line)
                    current_auuc = epoch_data.get("local_best_test_Target_Y_AUUC", -float('inf'))
                    if current_auuc > best_auuc:
                        best_auuc = current_auuc
                        best_epoch_metrics = epoch_data
        except Exception as e:
            continue
            
        if best_epoch_metrics is not None:
            # 将 list 形式的 head_hidden_dims 转换为 str，以便后面 groupby 分类
            head_dims = params.get("head_hidden_dims")
            head_dims_str = str(head_dims) if head_dims is not None else "None"
            
            # 提取你指定的关键参数
            trial_info = {
                "trial_id": best_epoch_metrics.get("trial_id", os.path.basename(root)),
                "head_hidden_dims": head_dims_str,
                "alpha_wool": params.get("conflict_alpha_wool"),
                "alpha_gold": params.get("conflict_alpha_gold"),
                "weight_decay": params.get("weight_decay"),
                "learning_rate": params.get("learning_rate"),
                "best_epoch": best_epoch_metrics.get("epoch"),
                "best_test_Y_AUUC": best_auuc,
                "best_test_C_AUUC": best_epoch_metrics.get("local_best_test_Target_C_AUUC"),
                "loss": best_epoch_metrics.get("loss")
            }
            all_trials_data.append(trial_info)

# %%
df = pd.DataFrame(all_trials_data)

if df.empty:
    print("没有找到有效的实验数据，请检查路径。")
else:
    print(f"成功解析了 {len(df)} 个实验 Trial。\n")
    
    # -------------------------------------------------------------
    # 维度 1: 按 head_hidden_dims 分类打印
    # -------------------------------------------------------------
    print("="*60)
    print(" 1. 按 head_hidden_dims 分类查看最佳表现 ")
    print("="*60)
    grouped_head = df.sort_values(by="best_test_Y_AUUC", ascending=False).groupby("head_hidden_dims")
    for name, group in grouped_head:
        top = group.iloc[0]
        print(f"Head 结构: {name} | 实验总数: {len(group)}")
        print(f"  -> 最佳最佳表现: Test Y AUUC = {top['best_test_Y_AUUC']:.6f} (Epoch {top['best_epoch']})")
        print(f"  -> 此时的其他参数: alpha_wool={top['alpha_wool']}, alpha_gold={top['alpha_gold']}, wd={top['weight_decay']}\n")

    # -------------------------------------------------------------
    # 维度 2: 按 conflict_alpha (wool & gold) 分类打印
    # -------------------------------------------------------------
    print("="*60)
    print(" 2. 按 conflict_alpha 权重分配分类查看最佳表现 ")
    print("="*60)
    grouped_alpha = df.sort_values(by="best_test_Y_AUUC", ascending=False).groupby(["alpha_wool", "alpha_gold"])
    for name, group in grouped_alpha:
        top = group.iloc[0]
        print(f"Alpha 配置: wool={name[0]}, gold={name[1]} | 实验总数: {len(group)}")
        print(f"  -> 最佳表现: Test Y AUUC = {top['best_test_Y_AUUC']:.6f}")
        print(f"  -> 此时的其他参数: Head={top['head_hidden_dims']}, wd={top['weight_decay']}\n")

    # -------------------------------------------------------------
    # 维度 3: 按 weight_decay 分类打印
    # -------------------------------------------------------------
    print("="*60)
    print(" 3. 按 weight_decay 分类查看最佳表现 ")
    print("="*60)
    grouped_wd = df.sort_values(by="best_test_Y_AUUC", ascending=False).groupby("weight_decay")
    for name, group in grouped_wd:
        top = group.iloc[0]
        print(f"Weight Decay: {name} | 实验总数: {len(group)}")
        print(f"  -> 最佳表现: Test Y AUUC = {top['best_test_Y_AUUC']:.6f}")
        print(f"  -> 此时的其他参数: Head={top['head_hidden_dims']}, alpha_wool={top['alpha_wool']}, alpha_gold={top['alpha_gold']}\n")

    # -------------------------------------------------------------
    # 详细透视表：联合看 Head 和 Alpha 对结果的交叉影响
    # -------------------------------------------------------------
    print("="*60)
    print(" 4. 联合透视表 (Head x Alpha_wool) 的最高 Test Y AUUC ")
    print("="*60)
    pivot_df = df.pivot_table(
        index="head_hidden_dims", 
        columns="alpha_wool", 
        values="best_test_Y_AUUC", 
        aggfunc="max"
    )
    print(pivot_df.to_string())

开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha_head/exp_v10_mix_both_head_mean
成功解析了 32 个实验 Trial。

 1. 按 head_hidden_dims 分类查看最佳表现 
Head 结构: [128, 64, 32] | 实验总数: 8
  -> 最佳最佳表现: Test Y AUUC = 0.904297 (Epoch 6)
  -> 此时的其他参数: alpha_wool=1.0, alpha_gold=1.0, wd=0.0001

Head 结构: [32, 32] | 实验总数: 8
  -> 最佳最佳表现: Test Y AUUC = 0.903964 (Epoch 3)
  -> 此时的其他参数: alpha_wool=10.0, alpha_gold=10.0, wd=1e-05

Head 结构: [32] | 实验总数: 8
  -> 最佳最佳表现: Test Y AUUC = 0.909678 (Epoch 5)
  -> 此时的其他参数: alpha_wool=10.0, alpha_gold=10.0, wd=0.01

Head 结构: [64, 32] | 实验总数: 8
  -> 最佳最佳表现: Test Y AUUC = 0.910698 (Epoch 7)
  -> 此时的其他参数: alpha_wool=1.0, alpha_gold=1.0, wd=1e-05

 2. 按 conflict_alpha 权重分配分类查看最佳表现 
Alpha 配置: wool=1.0, gold=1.0 | 实验总数: 16
  -> 最佳表现: Test Y AUUC = 0.910698
  -> 此时的其他参数: Head=[64, 32], wd=1e-05

Alpha 配置: wool=10.0, gold=10.0 | 实验总数: 16
  -> 最佳表现: Test Y AUUC = 0.909678
  -> 此时的其他参数: Head=[32], wd=0.01

 3. 按 weight_decay

In [2]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析脚本
# 逻辑：直接读取每个 Trial 的最后一行（最终 Epoch），取出 local_best_test_Target_Y_AUUC，
# 并根据 head_hidden_dims, alpha, weight_decay 进行分类对比。

# %%
import os
import json
import pandas as pd

# %%
# 定义基础搜索路径
base_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha_head/exp_v10_mix_both_head_mean"

print(f"开始解析路径: {base_path}")

# %%
all_trials_data = []

# 遍历目录
for root, dirs, files in os.walk(base_path):
    if "params.json" in files and "result.json" in files:
        params_path = os.path.join(root, "params.json")
        result_path = os.path.join(root, "result.json")
        
        # 1. 解析参数
        try:
            with open(params_path, 'r', encoding='utf-8') as f:
                params = json.load(f)
        except Exception:
            continue
            
        # 2. 定位最后一轮的结算指标
        last_epoch_data = None
        try:
            with open(result_path, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
                if lines:
                    # 直接获取最后一行（最后 Epoch）
                    last_epoch_data = json.loads(lines[-1])
        except Exception:
            continue
            
        if last_epoch_data is not None:
            # 转换 list 为 str 方便后续做分类聚合
            head_dims = params.get("head_hidden_dims")
            head_dims_str = str(head_dims) if head_dims is not None else "None"
            
            # 提取最终结算数据
            trial_info = {
                "trial_id": last_epoch_data.get("trial_id", os.path.basename(root)),
                "head_hidden_dims": head_dims_str,
                "alpha_wool": params.get("conflict_alpha_wool"),
                "alpha_gold": params.get("conflict_alpha_gold"),
                "weight_decay": params.get("weight_decay"),
                "final_epoch": last_epoch_data.get("epoch"),
                # 准确抓取最后结算时的 local best test 指标
                "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                "final_loss": last_epoch_data.get("loss")
            }
            all_trials_data.append(trial_info)

# %%
df = pd.DataFrame(all_trials_data)

if df.empty:
    print("没有找到任何有效的实验数据，请检查路径。")
else:
    print(f"成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # 核心超参数分类列表
    core_params = ["head_hidden_dims", "alpha_wool", "alpha_gold", "weight_decay"]
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合直接 Groupby 展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    # 按参数分组，并找出每组中最终 AUUC 最高的那个 Trial
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Head: {row['head_hidden_dims']} | alpha_wool: {row['alpha_wool']} | alpha_gold: {row['alpha_gold']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比（直观呈现各参数在最后阶段的战况）
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))

开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha_head/exp_v10_mix_both_head_mean
成功加载 32 个实验的最终结算数据。

 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: [64, 32] | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 1e-05
  -> 最终结算 Epoch: 17 (Loss: 0.0198)
  -> 最终 local_best_test_Y_AUUC: 0.910698
  -> 最终 local_best_test_C_AUUC: 0.824818
  -> Trial ID: 3be23_00002

[配置组合] Head: [64, 32] | alpha_wool: 10.0 | alpha_gold: 10.0 | wd: 1e-05
  -> 最终结算 Epoch: 15 (Loss: 0.0643)
  -> 最终 local_best_test_Y_AUUC: 0.906284
  -> 最终 local_best_test_C_AUUC: 0.821953
  -> Trial ID: 3be23_00003

[配置组合] Head: [128, 64, 32] | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 0.0001
  -> 最终结算 Epoch: 16 (Loss: 0.0191)
  -> 最终 local_best_test_Y_AUUC: 0.904297
  -> 最终 local_best_test_C_AUUC: 0.820527
  -> Trial ID: 3be23_00012

[配置组合] Head: [32, 32] | alpha_wool: 10.0 | alpha_gold: 10.0 | wd: 1e-05
  -> 最终结算 Epoch: 13 (Loss: 0.0658)
  -> 最终 local_best_test

In [3]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析脚本
# 逻辑：直接读取每个 Trial 的最后一行（最终 Epoch），取出 local_best_test_Target_Y_AUUC，
# 并根据 head_hidden_dims, alpha, weight_decay 进行分类对比。

# %%
import os
import json
import pandas as pd

# %%
# 定义基础搜索路径
base_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_all_same_alpha/exp_v10_mix_all/exp_v10_mix_all"

print(f"开始解析路径: {base_path}")

# %%
all_trials_data = []

# 遍历目录
for root, dirs, files in os.walk(base_path):
    if "params.json" in files and "result.json" in files:
        params_path = os.path.join(root, "params.json")
        result_path = os.path.join(root, "result.json")
        
        # 1. 解析参数
        try:
            with open(params_path, 'r', encoding='utf-8') as f:
                params = json.load(f)
        except Exception:
            continue
            
        # 2. 定位最后一轮的结算指标
        last_epoch_data = None
        try:
            with open(result_path, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
                if lines:
                    # 直接获取最后一行（最后 Epoch）
                    last_epoch_data = json.loads(lines[-1])
        except Exception:
            continue
            
        if last_epoch_data is not None:
            # 转换 list 为 str 方便后续做分类聚合
            head_dims = params.get("head_hidden_dims")
            head_dims_str = str(head_dims) if head_dims is not None else "None"
            
            # 提取最终结算数据
            trial_info = {
                "trial_id": last_epoch_data.get("trial_id", os.path.basename(root)),
                "head_hidden_dims": head_dims_str,
                "alpha_wool": params.get("conflict_alpha_wool"),
                "alpha_gold": params.get("conflict_alpha_gold"),
                "weight_decay": params.get("weight_decay"),
                "final_epoch": last_epoch_data.get("epoch"),
                # 准确抓取最后结算时的 local best test 指标
                "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                "final_loss": last_epoch_data.get("loss")
            }
            all_trials_data.append(trial_info)

# %%
df = pd.DataFrame(all_trials_data)

if df.empty:
    print("没有找到任何有效的实验数据，请检查路径。")
else:
    print(f"成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # 核心超参数分类列表
    core_params = ["head_hidden_dims", "alpha_wool", "alpha_gold", "weight_decay"]
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合直接 Groupby 展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    # 按参数分组，并找出每组中最终 AUUC 最高的那个 Trial
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Head: {row['head_hidden_dims']} | alpha_wool: {row['alpha_wool']} | alpha_gold: {row['alpha_gold']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比（直观呈现各参数在最后阶段的战况）
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))

开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_all_same_alpha/exp_v10_mix_all/exp_v10_mix_all


成功加载 84 个实验的最终结算数据。

 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 0.5 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 14 (Loss: 0.0456)
  -> 最终 local_best_test_Y_AUUC: 0.907201
  -> 最终 local_best_test_C_AUUC: 0.824569
  -> Trial ID: 69a98_00038

[配置组合] Head: None | alpha_wool: 0.5 | alpha_gold: 10.0 | wd: 1e-05
  -> 最终结算 Epoch: 14 (Loss: 0.0628)
  -> 最终 local_best_test_Y_AUUC: 0.907155
  -> 最终 local_best_test_C_AUUC: 0.818229
  -> Trial ID: 69a98_00034

[配置组合] Head: None | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 1e-05
  -> 最终结算 Epoch: 20 (Loss: 0.0166)
  -> 最终 local_best_test_Y_AUUC: 0.906704
  -> 最终 local_best_test_C_AUUC: 0.801980
  -> Trial ID: 69a98_00026

[配置组合] Head: None | alpha_wool: 0.1 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 14 (Loss: 0.0435)
  -> 最终 local_best_test_Y_AUUC: 0.905474
  -> 最终 local_best_test_C_AUUC: 0.826830
  -> Trial ID: 69a98_00003

[配置组合] Head: None | alpha_wool: 0.1 | alpha_gold: 10.0 | wd: 1e-05
  -> 最终结算 Epoch:

In [4]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析脚本
# 逻辑：直接读取每个 Trial 的最后一行（最终 Epoch），取出 local_best_test_Target_Y_AUUC，
# 并根据 head_hidden_dims, alpha, weight_decay 进行分类对比。

# %%
import os
import json
import pandas as pd

# %%
# 定义基础搜索路径
base_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha/exp_v10_mix_both_mean/exp_v10_mix_both_mean"

print(f"开始解析路径: {base_path}")

# %%
all_trials_data = []

# 遍历目录
for root, dirs, files in os.walk(base_path):
    if "params.json" in files and "result.json" in files:
        params_path = os.path.join(root, "params.json")
        result_path = os.path.join(root, "result.json")
        
        # 1. 解析参数
        try:
            with open(params_path, 'r', encoding='utf-8') as f:
                params = json.load(f)
        except Exception:
            continue
            
        # 2. 定位最后一轮的结算指标
        last_epoch_data = None
        try:
            with open(result_path, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
                if lines:
                    # 直接获取最后一行（最后 Epoch）
                    last_epoch_data = json.loads(lines[-1])
        except Exception:
            continue
            
        if last_epoch_data is not None:
            # 转换 list 为 str 方便后续做分类聚合
            head_dims = params.get("head_hidden_dims")
            head_dims_str = str(head_dims) if head_dims is not None else "None"
            
            # 提取最终结算数据
            trial_info = {
                "trial_id": last_epoch_data.get("trial_id", os.path.basename(root)),
                "head_hidden_dims": head_dims_str,
                "alpha_wool": params.get("conflict_alpha_wool"),
                "alpha_gold": params.get("conflict_alpha_gold"),
                "weight_decay": params.get("weight_decay"),
                "final_epoch": last_epoch_data.get("epoch"),
                # 准确抓取最后结算时的 local best test 指标
                "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                "final_loss": last_epoch_data.get("loss")
            }
            all_trials_data.append(trial_info)

# %%
df = pd.DataFrame(all_trials_data)

if df.empty:
    print("没有找到任何有效的实验数据，请检查路径。")
else:
    print(f"成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # 核心超参数分类列表
    core_params = ["head_hidden_dims", "alpha_wool", "alpha_gold", "weight_decay"]
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合直接 Groupby 展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    # 按参数分组，并找出每组中最终 AUUC 最高的那个 Trial
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Head: {row['head_hidden_dims']} | alpha_wool: {row['alpha_wool']} | alpha_gold: {row['alpha_gold']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比（直观呈现各参数在最后阶段的战况）
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))

开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha/exp_v10_mix_both_mean/exp_v10_mix_both_mean
成功加载 12 个实验的最终结算数据。

 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 0.001
  -> 最终结算 Epoch: 15 (Loss: 0.0214)
  -> 最终 local_best_test_Y_AUUC: 0.904882
  -> 最终 local_best_test_C_AUUC: 0.821955
  -> Trial ID: bd067_00007

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 0.01
  -> 最终结算 Epoch: 15 (Loss: 0.0523)
  -> 最终 local_best_test_Y_AUUC: 0.903416
  -> 最终 local_best_test_C_AUUC: 0.831175
  -> Trial ID: bd067_00010

[配置组合] Head: None | alpha_wool: 10.0 | alpha_gold: 10.0 | wd: 0.001
  -> 最终结算 Epoch: 13 (Loss: 0.0689)
  -> 最终 local_best_test_Y_AUUC: 0.901947
  -> 最终 local_best_test_C_AUUC: 0.833030
  -> Trial ID: bd067_00008

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 13 (Loss: 0.0312)
  -> 最终 local_best_test_Y_AUUC: 0.89

In [5]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析脚本
# 逻辑：直接读取每个 Trial 的最后一行（最终 Epoch），取出 local_best_test_Target_Y_AUUC，
# 并根据 head_hidden_dims, alpha, weight_decay 进行分类对比。

# %%
import os
import json
import pandas as pd

# %%
# 定义基础搜索路径
base_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha/exp_v10_mix_both_mean/exp_v10_mix_both_mean"

print(f"开始解析路径: {base_path}")

# %%
all_trials_data = []

# 遍历目录
for root, dirs, files in os.walk(base_path):
    if "params.json" in files and "result.json" in files:
        params_path = os.path.join(root, "params.json")
        result_path = os.path.join(root, "result.json")
        
        # 1. 解析参数
        try:
            with open(params_path, 'r', encoding='utf-8') as f:
                params = json.load(f)
        except Exception:
            continue
            
        # 2. 定位最后一轮的结算指标
        last_epoch_data = None
        try:
            with open(result_path, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
                if lines:
                    # 直接获取最后一行（最后 Epoch）
                    last_epoch_data = json.loads(lines[-1])
        except Exception:
            continue
            
        if last_epoch_data is not None:
            # 转换 list 为 str 方便后续做分类聚合
            head_dims = params.get("head_hidden_dims")
            head_dims_str = str(head_dims) if head_dims is not None else "None"
            
            # 提取最终结算数据
            trial_info = {
                "trial_id": last_epoch_data.get("trial_id", os.path.basename(root)),
                "head_hidden_dims": head_dims_str,
                "alpha_wool": params.get("conflict_alpha_wool"),
                "alpha_gold": params.get("conflict_alpha_gold"),
                "weight_decay": params.get("weight_decay"),
                "final_epoch": last_epoch_data.get("epoch"),
                # 准确抓取最后结算时的 local best test 指标
                "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                "final_loss": last_epoch_data.get("loss")
            }
            all_trials_data.append(trial_info)

# %%
df = pd.DataFrame(all_trials_data)

if df.empty:
    print("没有找到任何有效的实验数据，请检查路径。")
else:
    print(f"成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # 核心超参数分类列表
    core_params = ["head_hidden_dims", "alpha_wool", "alpha_gold", "weight_decay"]
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合直接 Groupby 展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    # 按参数分组，并找出每组中最终 AUUC 最高的那个 Trial
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Head: {row['head_hidden_dims']} | alpha_wool: {row['alpha_wool']} | alpha_gold: {row['alpha_gold']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比（直观呈现各参数在最后阶段的战况）
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))

开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha/exp_v10_mix_both_mean/exp_v10_mix_both_mean
成功加载 12 个实验的最终结算数据。

 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 0.001
  -> 最终结算 Epoch: 15 (Loss: 0.0214)
  -> 最终 local_best_test_Y_AUUC: 0.904882
  -> 最终 local_best_test_C_AUUC: 0.821955
  -> Trial ID: bd067_00007

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 0.01
  -> 最终结算 Epoch: 15 (Loss: 0.0523)
  -> 最终 local_best_test_Y_AUUC: 0.903416
  -> 最终 local_best_test_C_AUUC: 0.831175
  -> Trial ID: bd067_00010

[配置组合] Head: None | alpha_wool: 10.0 | alpha_gold: 10.0 | wd: 0.001
  -> 最终结算 Epoch: 13 (Loss: 0.0689)
  -> 最终 local_best_test_Y_AUUC: 0.901947
  -> 最终 local_best_test_C_AUUC: 0.833030
  -> Trial ID: bd067_00008

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 13 (Loss: 0.0312)
  -> 最终 local_best_test_Y_AUUC: 0.89

In [6]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析脚本
# 逻辑：直接读取每个 Trial 的最后一行（最终 Epoch），取出 local_best_test_Target_Y_AUUC，
# 并根据 head_hidden_dims, alpha, weight_decay 进行分类对比。

# %%
import os
import json
import pandas as pd

# %%
# 定义基础搜索路径
base_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha/exp_v10_mix_both"

print(f"开始解析路径: {base_path}")

# %%
all_trials_data = []

# 遍历目录
for root, dirs, files in os.walk(base_path):
    if "params.json" in files and "result.json" in files:
        params_path = os.path.join(root, "params.json")
        result_path = os.path.join(root, "result.json")
        
        # 1. 解析参数
        try:
            with open(params_path, 'r', encoding='utf-8') as f:
                params = json.load(f)
        except Exception:
            continue
            
        # 2. 定位最后一轮的结算指标
        last_epoch_data = None
        try:
            with open(result_path, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
                if lines:
                    # 直接获取最后一行（最后 Epoch）
                    last_epoch_data = json.loads(lines[-1])
        except Exception:
            continue
            
        if last_epoch_data is not None:
            # 转换 list 为 str 方便后续做分类聚合
            head_dims = params.get("head_hidden_dims")
            head_dims_str = str(head_dims) if head_dims is not None else "None"
            
            # 提取最终结算数据
            trial_info = {
                "trial_id": last_epoch_data.get("trial_id", os.path.basename(root)),
                "head_hidden_dims": head_dims_str,
                "alpha_wool": params.get("conflict_alpha_wool"),
                "alpha_gold": params.get("conflict_alpha_gold"),
                "weight_decay": params.get("weight_decay"),
                "final_epoch": last_epoch_data.get("epoch"),
                # 准确抓取最后结算时的 local best test 指标
                "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                "final_loss": last_epoch_data.get("loss")
            }
            all_trials_data.append(trial_info)

# %%
df = pd.DataFrame(all_trials_data)

if df.empty:
    print("没有找到任何有效的实验数据，请检查路径。")
else:
    print(f"成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # 核心超参数分类列表
    core_params = ["head_hidden_dims", "alpha_wool", "alpha_gold", "weight_decay"]
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合直接 Groupby 展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    # 按参数分组，并找出每组中最终 AUUC 最高的那个 Trial
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Head: {row['head_hidden_dims']} | alpha_wool: {row['alpha_wool']} | alpha_gold: {row['alpha_gold']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比（直观呈现各参数在最后阶段的战况）
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))

开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_both_same_alpha/exp_v10_mix_both
成功加载 15 个实验的最终结算数据。

 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 0.001
  -> 最终结算 Epoch: 14 (Loss: 0.0187)
  -> 最终 local_best_test_Y_AUUC: 0.904767
  -> 最终 local_best_test_C_AUUC: 0.817452
  -> Trial ID: 93c85_00011

[配置组合] Head: None | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 0.001
  -> 最终结算 Epoch: 15 (Loss: 0.0231)
  -> 最终 local_best_test_Y_AUUC: 0.901440
  -> 最终 local_best_test_C_AUUC: 0.823212
  -> Trial ID: 93c85_00012

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 25 (Loss: 0.0452)
  -> 最终 local_best_test_Y_AUUC: 0.900938
  -> 最终 local_best_test_C_AUUC: 0.834340
  -> Trial ID: 93c85_00003

[配置组合] Head: None | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 1e-05
  -> 最终结算 Epoch: 26 (Loss: 0.0163)
  -> 最终 local_best_test_Y_AUUC: 0.899309
  -> 最终 local_best_test

In [7]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析函数
# 逻辑：传入 base_path，直接定位每个 Trial 的最后一行（最终 Epoch），
# 提取出最终的 local_best_test_Target_Y_AUUC，并按关键参数分类打印。

# %%
import os
import json
import pandas as pd

# %%
def analyze_ray_results(base_path):
    """
    遍历指定的 Ray 结果路径，按 head_hidden_dims, alpha, weight_decay 分类对比最后 Epoch 的结算指标。
    
    Args:
        base_path (str): Ray 实验结果的根目录或特定实验版本的目录
    """
    print(f"===========================================================")
    print(f" 开始解析路径: {base_path}")
    print(f"===========================================================")
    
    all_trials_data = []

    # 遍历目录
    for root, dirs, files in os.walk(base_path):
        if "params.json" in files and "result.json" in files:
            params_path = os.path.join(root, "params.json")
            result_path = os.path.join(root, "result.json")
            
            # 1. 解析参数
            try:
                with open(params_path, 'r', encoding='utf-8') as f:
                    params = json.load(f)
            except Exception:
                continue
                
            # 2. 定位最后一轮的结算指标
            last_epoch_data = None
            try:
                with open(result_path, 'r', encoding='utf-8') as f:
                    lines = [line.strip() for line in f if line.strip()]
                    if lines:
                        # 直接获取最后一行（最后 Epoch）
                        last_epoch_data = json.loads(lines[-1])
            except Exception:
                continue
                
            if last_epoch_data is not None:
                # 转换 list 为 str 方便后续做分类聚合
                head_dims = params.get("head_hidden_dims")
                head_dims_str = str(head_dims) if head_dims is not None else "None"
                
                # 提取最终结算数据
                trial_info = {
                    "trial_id": last_epoch_data.get("trial_id", os.path.basename(root)),
                    "head_hidden_dims": head_dims_str,
                    "alpha_wool": params.get("conflict_alpha_wool"),
                    "alpha_gold": params.get("conflict_alpha_gold"),
                    "weight_decay": params.get("weight_decay"),
                    "final_epoch": last_epoch_data.get("epoch"),
                    # 准确抓取最后结算时的 local best test 指标
                    "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                    "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                    "final_loss": last_epoch_data.get("loss")
                }
                all_trials_data.append(trial_info)

    # 3. 统计并打印结果
    df = pd.DataFrame(all_trials_data)

    if df.empty:
        print("❌ 没有找到任何有效的实验数据，请检查传入的路径是否正确。\n")
        return None
        
    print(f"📊 成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # 核心超参数分类列表
    core_params = ["head_hidden_dims", "alpha_wool", "alpha_gold", "weight_decay"]
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    # 按参数分组，并找出每组中最终 AUUC 最高的那个 Trial
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Head: {row['head_hidden_dims']} | alpha_wool: {row['alpha_wool']} | alpha_gold: {row['alpha_gold']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 📋 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))
    print("\n")
    
    return df

# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_solo_wool/exp_v10_solo_wool_mean"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_solo_wool/exp_v10_solo_wool_mean


📊 成功加载 20 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 0.0 | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0134)
  -> 最终 local_best_test_Y_AUUC: 0.904159
  -> 最终 local_best_test_C_AUUC: 0.818484
  -> Trial ID: a5612_00003

[配置组合] Head: None | alpha_wool: 0.1 | alpha_gold: 0.0 | wd: 0.001
  -> 最终结算 Epoch: 16 (Loss: 0.0145)
  -> 最终 local_best_test_Y_AUUC: 0.901164
  -> 最终 local_best_test_C_AUUC: 0.833517
  -> Trial ID: a5612_00010

[配置组合] Head: None | alpha_wool: 0.1 | alpha_gold: 0.0 | wd: 0.0001
  -> 最终结算 Epoch: 18 (Loss: 0.0125)
  -> 最终 local_best_test_Y_AUUC: 0.901137
  -> 最终 local_best_test_C_AUUC: 0.825455
  -> Trial ID: a5612_00005

[配置组合] Head: None | alpha_wool: 1.0 | alpha_gold: 0.0 | wd: 0.0001
  -> 最终结算 Epoch: 16 (Loss: 0.0126)
  -> 最终 local_best_test_Y_AUUC: 0.900101
  -> 最终 local_best_test_C_AUUC: 0.833626
  -> Trial ID: a5612_00007

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 0.0 | wd: 0.0001
  -> 最终结算 E

In [8]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_solo_gold/exp_v10_solo_gold/exp_v10_solo_gold"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_solo_gold/exp_v10_solo_gold/exp_v10_solo_gold
📊 成功加载 23 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 0.0 | alpha_gold: 10.0 | wd: 0.0001
  -> 最终结算 Epoch: 13 (Loss: 0.0693)
  -> 最终 local_best_test_Y_AUUC: 0.903411
  -> 最终 local_best_test_C_AUUC: 0.831450
  -> Trial ID: 41052_00009

[配置组合] Head: None | alpha_wool: 0.0 | alpha_gold: 5.0 | wd: 0.001
  -> 最终结算 Epoch: 14 (Loss: 0.0453)
  -> 最终 local_best_test_Y_AUUC: 0.903021
  -> 最终 local_best_test_C_AUUC: 0.819235
  -> Trial ID: 41052_00013

[配置组合] Head: None | alpha_wool: 0.0 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 15 (Loss: 0.0384)
  -> 最终 local_best_test_Y_AUUC: 0.902247
  -> 最终 local_best_test_C_AUUC: 0.835677
  -> Trial ID: 41052_00003

[配置组合] Head: None | alpha_wool: 0.0 | alpha_gold: 10.0 | wd: 0.001
  -> 最终结算 Epoch: 14 (Loss: 0.0658)
  -> 最终 local_best_test_Y_AUUC: 0.902244
  -> 最

In [9]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_solo_gold/exp_v10_solo_gold_mean/exp_v10_solo_gold_mean"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_solo_gold/exp_v10_solo_gold_mean/exp_v10_solo_gold_mean
📊 成功加载 20 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 0.0 | alpha_gold: 1.0 | wd: 0.0001
  -> 最终结算 Epoch: 15 (Loss: 0.0182)
  -> 最终 local_best_test_Y_AUUC: 0.908165
  -> 最终 local_best_test_C_AUUC: 0.815421
  -> Trial ID: ef3e8_00007

[配置组合] Head: None | alpha_wool: 0.0 | alpha_gold: 10.0 | wd: 0.001
  -> 最终结算 Epoch: 14 (Loss: 0.0700)
  -> 最终 local_best_test_Y_AUUC: 0.905017
  -> 最终 local_best_test_C_AUUC: 0.833369
  -> Trial ID: ef3e8_00014

[配置组合] Head: None | alpha_wool: 0.0 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 14 (Loss: 0.0435)
  -> 最终 local_best_test_Y_AUUC: 0.904871
  -> 最终 local_best_test_C_AUUC: 0.823768
  -> Trial ID: ef3e8_00003

[配置组合] Head: None | alpha_wool: 0.0 | alpha_gold: 10.0 | wd: 0.01
  -> 最终结算 Epoch: 13 (Loss: 0.0761)
  -> 最终 local_best_test_Y_AUUC: 0.9034

In [10]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_all_same_alpha_head_0710/exp_y_pure_v10_all_same_alpha_head_0710/exp_y_pure_v10_all_same_alpha_head_0710"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_all_same_alpha_head_0710/exp_y_pure_v10_all_same_alpha_head_0710/exp_y_pure_v10_all_same_alpha_head_0710
📊 成功加载 30 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: [32] | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0134)
  -> 最终 local_best_test_Y_AUUC: 0.906664
  -> 最终 local_best_test_C_AUUC: 0.815630
  -> Trial ID: 151dd_00001

[配置组合] Head: [32] | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 0.0001
  -> 最终结算 Epoch: 18 (Loss: 0.0126)
  -> 最终 local_best_test_Y_AUUC: 0.904860
  -> 最终 local_best_test_C_AUUC: 0.818359
  -> Trial ID: 151dd_00016

[配置组合] Head: [32, 16] | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 0.0001
  -> 最终结算 Epoch: 25 (Loss: 0.0124)
  -> 最终 local_best_test_Y_AUUC: 0.904623
  -> 最终 local_best_test_C_AUUC: 0.821519
  -> Trial ID: 151dd_00023

[配置组合] Head: [32] | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 0.0001
  -> 最终结算 Epoch: 20

In [11]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_all_same_alpha_head_0710/exp_y_pure_v10_all_same_alpha_head_0710_res_2_layer_head_search"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_all_same_alpha_head_0710/exp_y_pure_v10_all_same_alpha_head_0710_res_2_layer_head_search
📊 成功加载 30 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: [32] | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 0.0001
  -> 最终结算 Epoch: 30 (Loss: 0.0122)
  -> 最终 local_best_test_Y_AUUC: 0.922116
  -> 最终 local_best_test_C_AUUC: 0.823849
  -> Trial ID: 141ec_00015

[配置组合] Head: [16] | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 1e-05
  -> 最终结算 Epoch: 16 (Loss: 0.0130)
  -> 最终 local_best_test_Y_AUUC: 0.913983
  -> 最终 local_best_test_C_AUUC: 0.827496
  -> Trial ID: 141ec_00013

[配置组合] Head: [64, 32] | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 0.0001
  -> 最终结算 Epoch: 19 (Loss: 0.0131)
  -> 最终 local_best_test_Y_AUUC: 0.912334
  -> 最终 local_best_test_C_AUUC: 0.807510
  -> Trial ID: 141ec_00020

[配置组合] Head: [16] | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 0.0001
  -> 最终结算 Epoch: 18 (Loss: 0.0127)


In [12]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0710/y_ours_s4_conflict_0710n/y_ours_s4_conflict_0710n"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0710/y_ours_s4_conflict_0710n/y_ours_s4_conflict_0710n
📊 成功加载 9 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0160)
  -> 最终 local_best_test_Y_AUUC: 0.905981
  -> 最终 local_best_test_C_AUUC: 0.818934
  -> Trial ID: d2962_00002

[配置组合] Head: None | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 0.0001
  -> 最终结算 Epoch: 15 (Loss: 0.0207)
  -> 最终 local_best_test_Y_AUUC: 0.905654
  -> 最终 local_best_test_C_AUUC: 0.821452
  -> Trial ID: d2962_00003

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 0.0001
  -> 最终结算 Epoch: 15 (Loss: 0.0374)
  -> 最终 local_best_test_Y_AUUC: 0.902570
  -> 最终 local_best_test_C_AUUC: 0.821288
  -> Trial ID: d2962_00004

[配置组合] Head: None | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 1e-05
  -> 最终结算 Epoch: 15 (Loss: 0.0179)
  -> 最终 local_best_test_Y_AUUC: 0.901669
  -> 最终 

In [13]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0710_alpha0"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0710_alpha0
📊 成功加载 20 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 0 | alpha_gold: 0 | wd: 0.0001
  -> 最终结算 Epoch: 21 (Loss: 0.0124)
  -> 最终 local_best_test_Y_AUUC: 0.906858
  -> 最终 local_best_test_C_AUUC: 0.815501
  -> Trial ID: aa49b_00008

[配置组合] Head: [16] | alpha_wool: 0 | alpha_gold: 0 | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0131)
  -> 最终 local_best_test_Y_AUUC: 0.904734
  -> 最终 local_best_test_C_AUUC: 0.830424
  -> Trial ID: aa49b_00004

[配置组合] Head: [32] | alpha_wool: 0 | alpha_gold: 0 | wd: 0.0001
  -> 最终结算 Epoch: 30 (Loss: 0.0123)
  -> 最终 local_best_test_Y_AUUC: 0.902968
  -> 最终 local_best_test_C_AUUC: 0.834281
  -> Trial ID: aa49b_00005

[配置组合] Head: [32, 16] | alpha_wool: 0 | alpha_gold: 0 | wd: 0.0001
  -> 最终结算 Epoch: 20 (Loss: 0.0127)
  -> 最终 local_best_test_Y_AUUC: 0.900396
  -> 最终 local_best_test_C_AUUC: 0.825746
  -> Trial ID: aa49b

In [15]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha0_temp10"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha0_temp10
📊 成功加载 20 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: [16] | alpha_wool: 0 | alpha_gold: 0 | wd: 0.001
  -> 最终结算 Epoch: 18 (Loss: 0.0148)
  -> 最终 local_best_test_Y_AUUC: 0.910731
  -> 最终 local_best_test_C_AUUC: 0.817845
  -> Trial ID: a22ba_00014

[配置组合] Head: [64, 32] | alpha_wool: 0 | alpha_gold: 0 | wd: 0.0001
  -> 最终结算 Epoch: 19 (Loss: 0.0126)
  -> 最终 local_best_test_Y_AUUC: 0.902147
  -> 最终 local_best_test_C_AUUC: 0.817370
  -> Trial ID: a22ba_00006

[配置组合] Head: [32, 16] | alpha_wool: 0 | alpha_gold: 0 | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0128)
  -> 最终 local_best_test_Y_AUUC: 0.902117
  -> 最终 local_best_test_C_AUUC: 0.825679
  -> Trial ID: a22ba_00002

[配置组合] Head: [32] | alpha_wool: 0 | alpha_gold: 0 | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0125)
  -> 最终 local_best_test_Y_AUUC: 0.900879
  -> 最终 local_best_test_C_AUUC: 0.826996
  -> Trial 

In [16]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha0_temp20"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha0_temp20
📊 成功加载 20 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: [64, 32] | alpha_wool: 0 | alpha_gold: 0 | wd: 0.0001
  -> 最终结算 Epoch: 19 (Loss: 0.0125)
  -> 最终 local_best_test_Y_AUUC: 0.902954
  -> 最终 local_best_test_C_AUUC: 0.822833
  -> Trial ID: ca3ca_00006

[配置组合] Head: None | alpha_wool: 0 | alpha_gold: 0 | wd: 0.001
  -> 最终结算 Epoch: 16 (Loss: 0.0140)
  -> 最终 local_best_test_Y_AUUC: 0.901810
  -> 最终 local_best_test_C_AUUC: 0.834880
  -> Trial ID: ca3ca_00013

[配置组合] Head: [32, 16] | alpha_wool: 0 | alpha_gold: 0 | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0130)
  -> 最终 local_best_test_Y_AUUC: 0.901042
  -> 最终 local_best_test_C_AUUC: 0.814452
  -> Trial ID: ca3ca_00002

[配置组合] Head: [32, 16] | alpha_wool: 0 | alpha_gold: 0 | wd: 0.001
  -> 最终结算 Epoch: 18 (Loss: 0.0146)
  -> 最终 local_best_test_Y_AUUC: 0.900646
  -> 最终 local_best_test_C_AUUC: 0.833549
  -> Tr

In [19]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha_search_temp1_20"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha_search_temp1_20
📊 成功加载 200 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: [32] | alpha_wool: 10.0 | alpha_gold: 10.0 | wd: 0.01
  -> 最终结算 Epoch: 11 (Loss: 0.0390)
  -> 最终 local_best_test_Y_AUUC: 0.913785
  -> 最终 local_best_test_C_AUUC: 0.830434
  -> Trial ID: 82204_00093

[配置组合] Head: [32] | alpha_wool: 0.1 | alpha_gold: 0.1 | wd: 1e-05
  -> 最终结算 Epoch: 21 (Loss: 0.0132)
  -> 最终 local_best_test_Y_AUUC: 0.912299
  -> 最终 local_best_test_C_AUUC: 0.803177
  -> Trial ID: 82204_00019

[配置组合] Head: [32] | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 1e-05
  -> 最终结算 Epoch: 19 (Loss: 0.0162)
  -> 最终 local_best_test_Y_AUUC: 0.910768
  -> 最终 local_best_test_C_AUUC: 0.829934
  -> Trial ID: 82204_00016

[配置组合] Head: None | alpha_wool: 1.0 | alpha_gold: 1.0 | wd: 1e-05
  -> 最终结算 Epoch: 17 (Loss: 0.0192)
  -> 最终 local_best_test_Y_AUUC: 0.909392
  -> 最终 local_best_test_C_AUUC: 0.8

In [20]:
# %%
# 在这里修改你的 base 路径即可调用
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha_search_temp10"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha_search_temp10
📊 成功加载 20 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Head: None | alpha_wool: 10.0 | alpha_gold: 10.0 | wd: 0.001
  -> 最终结算 Epoch: 11 (Loss: 0.0603)
  -> 最终 local_best_test_Y_AUUC: 0.909863
  -> 最终 local_best_test_C_AUUC: 0.828768
  -> Trial ID: 43361_00013

[配置组合] Head: None | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 0.001
  -> 最终结算 Epoch: 16 (Loss: 0.0181)
  -> 最终 local_best_test_Y_AUUC: 0.904389
  -> 最终 local_best_test_C_AUUC: 0.835063
  -> Trial ID: 43361_00011

[配置组合] Head: None | alpha_wool: 0.5 | alpha_gold: 0.5 | wd: 0.0001
  -> 最终结算 Epoch: 18 (Loss: 0.0171)
  -> 最终 local_best_test_Y_AUUC: 0.902013
  -> 最终 local_best_test_C_AUUC: 0.840058
  -> Trial ID: 43361_00006

[配置组合] Head: None | alpha_wool: 5.0 | alpha_gold: 5.0 | wd: 0.001
  -> 最终结算 Epoch: 14 (Loss: 0.0367)
  -> 最终 local_best_test_Y_AUUC: 0.901127
  -> 最终 local_best_test_C_AUUC: 0.82